# RAG System with RAGBench - Using Ingestion Layer and Strategy Classes

This notebook demonstrates the complete RAG pipeline using:
- Ingestion Layer for loading and parsing RAGBench datasets
- Strategy Factory for creating strategies
- RAG Config for configuration
- Complete RAG Pipeline for orchestration

## 1. Setup & Dependencies

In [1]:
!pip install datasets faiss-cpu sentence-transformers torch groq python-dotenv nltk -q

zsh:1: /Users/adityanarayan/aiml-talentsprint/rag-capstone-project/rag-foundry/venv/bin/pip: bad interpreter: /Users/adityanarayan/aiml-talentsprint/rag-capstone-project/real-world-rag-system/venv/bin/python: no such file or directory

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: /Users/adityanarayan/.pyenv/versions/3.11.9/bin/python -m pip install --upgrade pip


## 2. Imports

In [2]:
import os
import json
import pandas as pd
from typing import List, Dict, Any
from dotenv import load_dotenv
from groq import Groq
import numpy as np

# Load environment variables
load_dotenv(override=True)

# Get API keys
HF_TOKEN = os.getenv("HF_TOKEN")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print(f"HuggingFace token loaded: {bool(HF_TOKEN)}")
print(f"Groq API key loaded: {bool(GROQ_API_KEY)}")

HuggingFace token loaded: True
Groq API key loaded: True


## 3. Load Configuration

In [6]:
from rag.config.loader import ConfigLoader

# Load the high quality config (with reranking) from its YAML file.
# Providers are built automatically from `config.providers` when the
# pipeline is created, resolving credentials from each provider's `api_key_env`.
CONFIG_PATH = "config/rag_config_high_quality_small.yaml"
config = ConfigLoader.load(CONFIG_PATH)


print("Configuration:")
print(f"  Name:       {config.name}")
print(f"  Mode:       {config.mode}")
print(f"Providers:")
for provider_name, provider_config in config.providers.items():
    print(f"  {provider_name}: {provider_config.type.value} (api_key_env={provider_config.api_key_env})")
print(f"  Chunking:   {config.chunking.type.value}")
print(f"  Embedding:  {config.embedding.type.value} ({config.embedding.config.model_name})")
print(f"  Retrieval Pipeline: ")
for search in config.retrieval.search.searches:
    print(f"    - {search.type.value} (top_k={search.config.top_k})")
print(f"    - {config.retrieval.fusion} ")
print(f"    - {config.retrieval.rerank.type.value} (top_k={config.retrieval.rerank.config.top_k})")
print(f"  Generation: {config.generation.strategy.value} (provider={config.generation.provider}, model={config.generation.config.model})")
print(f"  Evaluation: {config.evaluation.type.value} (provider={config.evaluation.provider})")

Configuration:
  Name:       high_quality_small
  Mode:       Mode.DEV
Providers:
  groq: groq (api_key_env=GROQ_API_KEY)
  Chunking:   sentence
  Embedding:  sentence_transformer (BAAI/bge-large-en-v1.5)
  Retrieval Pipeline: 
    - dense (top_k=20)
    - None 
    - cross_encoder (top_k=5)
  Generation: default (provider=groq, model=llama-3.1-8b-instant)
  Evaluation: trace (provider=groq)


## 4. Load Data Using Data Loader

In [7]:
## 4. Load Data Using Ingestion Layer
from ingestion import (
    HuggingFaceLoader,
    ParserFactory,
    ParserType,
    DataProcessor,
    DatasetLoadingConfig
)

# Create loader for CovidQA dataset using generic HuggingFaceLoader
dataset_loading_config = DatasetLoadingConfig(limit=246)  # Full dataset
loader = HuggingFaceLoader(
    dataset_name="galileo-ai/ragbench",
    subset="covidqa",
    split="test",
    config=dataset_loading_config
)

# Load raw data
print("📥 Loading RAGBench CovidQA dataset using HuggingFaceLoader...")
raw_data = loader.load()
print(f"✅ Loaded {len(raw_data)} samples\n")

# Get dataset info
info = loader.info()
print(f"Dataset Info:")
print(f"  Source: {info['source']}")
print(f"  Dataset: {info['dataset_name']}")
print(f"  Subset: {info['subset']}")
print(f"  Split: {info['split']}")
print(f"  Samples: {info['num_samples']}")
print(f"  Keys: {len(info['keys'])} fields\n")

# Inspect first sample
first_sample = loader.load_sample(0)
print(f"First Sample:")
print(f"  Question: {first_sample['question'][:100]}...")
print(f"  Documents: {len(first_sample['documents'])}")
print(f"  Has ground truth scores: {bool(first_sample.get('relevance_score'))}")

/Users/adityanarayan/aiml-talentsprint/rag-capstone-project/rag-foundry/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📥 Loading RAGBench CovidQA dataset using HuggingFaceLoader...
Loading HuggingFace dataset: galileo-ai/ragbench/covidqa (test)...
Loaded 246 samples
✅ Loaded 246 samples

Dataset Info:
  Source: huggingface
  Dataset: galileo-ai/ragbench
  Subset: covidqa
  Split: test
  Samples: 246
  Keys: 26 fields

First Sample:
  Question: Which viruses may not cause prolonged inflammation due to strong induction of antiviral clearance?...
  Documents: 4
  Has ground truth scores: True


## 5. Parse Documents

In [8]:
## 5. Parse Documents Using Ingestion Layer
# Create parser strategy
print("📄 Creating parser strategy...")
parser = ParserFactory.create_parser(ParserType.TITLE_PASSAGE)
print(f"✅ Parser created: {parser.__class__.__name__}\n")

# Create data processor with parser strategy
processor = DataProcessor(parser_strategy=parser)

# Process dataset into canonical Document objects
print("📄 Processing documents...")
documents = processor.process_dataset(raw_data)
print(f"✅ Processed {len(documents)} documents\n")

# Print sample
sample_doc = documents[0]
print(f"Sample RAG Document:")
print(f"  Title: {sample_doc.title[:60]}...")
print(f"  Content: {sample_doc.content[:100]}...")
print(f"  Metadata: {sample_doc.metadata}")

📄 Creating parser strategy...
✅ Parser created: TitlePassageParser

📄 Processing documents...
✅ Processed 984 documents

Sample RAG Document:
  Title: Type I Interferon Receptor Deficiency in Dendritic Cells Fac...
  Content: successful treatment for HCV serves to circumvent the viral inhibition of IFN induction. Thus, HCV m...
  Metadata: {'doc_id': 'sample_0_doc_0', 'sample_index': 0, 'source': 'ragbench', 'parser_type': 'title_passage'}


## 6. Models Are Config-Driven

Both the embedding model and the reranking model are created inside the pipeline from the RAG config (`config.embedding` / `config.reranker`). No manual model loading is required, and no API clients are passed in: providers are built automatically from `config.providers`, resolving credentials from each provider's `api_key_env`.

In [9]:
# Both the embedding model and the reranking model are created inside the
# pipeline from the RAG config (config.embedding / config.reranker).
# No manual model loading or API clients are needed here -- providers are
# built automatically from config.providers when the pipeline is created.
print("Embedding + reranking models will be built by the pipeline from config.")

Embedding + reranking models will be built by the pipeline from config.


## 7. Initialize RAG Pipeline

In [11]:
from rag.pipeline.rag_pipeline import RAGPipeline

# The pipeline is a pure orchestrator. It builds every strategy from the config
# object and registers providers automatically from config.providers -- no
# strategy-specific parameters or API clients are passed in here.
print("Initializing RAG Pipeline...")
pipeline = RAGPipeline(config)
print("Pipeline initialized\n")

# Verify strategies were created
print("Strategies created:")
print(f"  Chunker:   {pipeline.chunker.__class__.__name__}")
print(f"  Embedder:  {pipeline.embedder.__class__.__name__}")
print(f"  Retriever: {pipeline.retriever.__class__.__name__}")
print(f"  Generator: {pipeline.generator.__class__.__name__}")
print(f"  Evaluator: {pipeline.evaluator.__class__.__name__}")

2026-06-28 00:40:29,072 DEBUG rag.pipeline.rag_pipeline: Initializing RAG pipeline in dev mode


Initializing RAG Pipeline...
Pipeline initialized

Strategies created:
  Chunker:   SentenceChunkingStrategy
  Embedder:  SentenceTransformerEmbeddingStrategy
  Retriever: RetrievalPipeline
  Generator: DefaultGenerationStrategy
  Evaluator: TRACeEvaluationStrategy


## 8. Build Vector Index

In [12]:
# Build index - this will chunk, embed, and store
print("🏗️  Building vector index...")
pipeline.build_index(documents)
print(f"✅ Index built and ready for querying")

2026-06-28 00:40:42,794 INFO rag.pipeline.rag_pipeline: Processing 984 documents...
2026-06-28 00:40:42,801 INFO rag.pipeline.rag_pipeline: [Chunk Cache] HIT key=a7c984e496237e69870e35aebde38527420d7c2e4e6cb3e3beb827e63656ab64
2026-06-28 00:40:42,802 DEBUG rag.pipeline.rag_pipeline: Chunk cache key=a7c984e496237e69870e35aebde38527420d7c2e4e6cb3e3beb827e63656ab64 (984 chunks)
2026-06-28 00:40:42,811 INFO rag.pipeline.rag_pipeline: [Embedding Cache] HIT key=53fa1b16676490718b5d1ab49d647571ed4624fa8eb217ab4b4184e4d027910c
2026-06-28 00:40:42,812 DEBUG rag.pipeline.rag_pipeline: Embedding cache key=53fa1b16676490718b5d1ab49d647571ed4624fa8eb217ab4b4184e4d027910c (984 vectors)
2026-06-28 00:40:42,818 INFO rag.pipeline.rag_pipeline: [Index Cache] HIT key=b3c3ecceb4ad62ed8cd0d0f0417c9710ca99ba2f8283a466b4d103076ac88bd8
2026-06-28 00:40:42,818 DEBUG rag.pipeline.rag_pipeline: Index cache key=b3c3ecceb4ad62ed8cd0d0f0417c9710ca99ba2f8283a466b4d103076ac88bd8
2026-06-28 00:40:42,818 INFO rag.pipel

🏗️  Building vector index...
✅ Index built and ready for querying


## 9. Test Single Query

In [13]:
# Test with first sample
test_sample = raw_data[0]
test_query = test_sample['question']

print(f"❓ Test Query:")
print(f"   {test_query}\n")

# Run query through pipeline
result = pipeline.query(test_query)

# Display results
print(f"\n📊 Query Results:")
print(f"\nRetrieved Documents: {len(result['retrieved_docs'])}")
for i, doc in enumerate(result['retrieved_docs'][:3], 1):
    print(f"  {i}. {doc['metadata']['title'][:60]}...")

print(f"\n💬 Generated Response:")
print(result['response'][:300] + "...\n")

print(f"📈 TRACe Scores:")
scores = result['scores']
print(f"  Relevance:    {scores['relevance_score']:.4f}")
print(f"  Utilization:  {scores['utilization_score']:.4f}")
print(f"  Completeness: {scores['completeness_score']:.4f}")
print(f"  Adherence:    {scores['adherence_score']}")

2026-06-28 00:40:46,082 INFO rag.pipeline.rag_pipeline: Query: Which viruses may not cause prolonged inflammation due to strong induction of antiviral clearance?
2026-06-28 00:40:46,083 DEBUG rag.pipeline.rag_pipeline: Retrieving documents...


❓ Test Query:
   Which viruses may not cause prolonged inflammation due to strong induction of antiviral clearance?



2026-06-28 00:40:48,191 DEBUG rag.pipeline.rag_pipeline: Retrieved 5 documents
2026-06-28 00:40:48,192 DEBUG rag.pipeline.rag_pipeline: Generating response...
2026-06-28 00:40:48,701 DEBUG rag.pipeline.rag_pipeline: Response generated (700 chars)
2026-06-28 00:40:48,701 DEBUG rag.pipeline.rag_pipeline: Evaluating response...
2026-06-28 00:40:49,681 DEBUG rag.pipeline.rag_pipeline: Evaluation complete



📊 Query Results:

Retrieved Documents: 5
  1. Type I Interferon Receptor Deficiency in Dendritic Cells Fac...
  2. Depletion of Alveolar Macrophages Does Not Prevent Hantaviru...
  3. Confounding roles for type I interferons during bacterial an...

💬 Generated Response:
Based on the given context, it seems that the viruses which may not cause prolonged inflammation due to their strong induction of antiviral clearance are:

1. Hantavirus (Document 4) - The protracted incubation period associated with hantavirus disease allows the host to mount a mature immune respon...

📈 TRACe Scores:
  Relevance:    0.9130
  Utilization:  0.1739
  Completeness: 0.1905
  Adherence:    False


## 10. Generate Detailed Report with ReportGenerator

The `ReportGenerator` is a thin orchestrator over pluggable `ReportStrategy`
implementations. We use the bundled `DetailedQueryReportStrategy`, which builds:

- a **per-query table**: query, retrieved documents as an array **with all
  retrieval scores**, the generated answer, and every TRACe score next to its
  dataset ground truth plus the per-query deviation; and
- an **aggregate table**: per TRACe metric, the mean score, mean ground truth,
  standard deviation of each, and mean absolute error vs. ground truth.

It reuses the `pipeline`, `config`, and `raw_data` already built above. New
report formats can be added by implementing `ReportStrategy` and registering it
with the generator — no change to `ReportGenerator` itself.

In [14]:
from rag.models.pipeline_run_result import PipelineRunResult
from rag.models.query_result import QueryResult

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

# Number of queries to include in the report.
NUM_REPORT_QUERIES = 5

records = []
for sample in raw_data[:NUM_REPORT_QUERIES]:
    query = sample["question"]
    result = pipeline.query(query)  # retrieved_docs carry all retrieval scores
    records.append(
        QueryResult(
            query=query,
            retrieved_docs=result["retrieved_docs"],
            answer=result["response"],
            metadata={
                "predicted_scores": result["scores"],
                "ground_truth_scores": {
                    "relevance_score": sample.get("relevance_score", 0.0),
                    "utilization_score": sample.get("utilization_score", 0.0),
                    "completeness_score": sample.get("completeness_score", 0.0),
                    "adherence_score": sample.get("adherence_score", False),
                },
            },
            
        )
    )

run = PipelineRunResult(
    records=records,
    config=config
)

2026-06-28 00:40:56,234 INFO rag.pipeline.rag_pipeline: Query: Which viruses may not cause prolonged inflammation due to strong induction of antiviral clearance?
2026-06-28 00:40:56,234 DEBUG rag.pipeline.rag_pipeline: Retrieving documents...
2026-06-28 00:40:57,796 DEBUG rag.pipeline.rag_pipeline: Retrieved 5 documents
2026-06-28 00:40:57,798 DEBUG rag.pipeline.rag_pipeline: Generating response...
2026-06-28 00:40:58,218 DEBUG rag.pipeline.rag_pipeline: Response generated (848 chars)
2026-06-28 00:40:58,219 DEBUG rag.pipeline.rag_pipeline: Evaluating response...
2026-06-28 00:41:23,403 DEBUG rag.pipeline.rag_pipeline: Evaluation complete
2026-06-28 00:41:23,404 INFO rag.pipeline.rag_pipeline: Query: When was the first case of COVID-19 identified?
2026-06-28 00:41:23,404 DEBUG rag.pipeline.rag_pipeline: Retrieving documents...
2026-06-28 00:41:24,852 DEBUG rag.pipeline.rag_pipeline: Retrieved 5 documents
2026-06-28 00:41:24,853 DEBUG rag.pipeline.rag_pipeline: Generating response...
20

In [15]:
from reporting.detailed_report import DetailedQueryReportStrategy
from reporting.report_generator import ReportGenerator

generator = ReportGenerator(DetailedQueryReportStrategy())
print("Available report strategies:", generator.strategies)

report = generator.generate(run);
report.save_json(f"reports/{run.config.name}.json");

Available report strategies: ['detailed_query']


In [16]:
report.display()

# RAG Multi-Config Evaluation Report

_Strategy: detailed_query_

## Config: `high_quality_small`

**name**: high_quality_small  •  **mode**: dev  •  **providers**: {'groq': {'type': 'groq', 'api_key_env': 'GROQ_API_KEY', 'params': {'cooldown_seconds': 60}}}  •  **chunking**: {'type': 'sentence', 'config': {'max_words': 100, 'overlap_sentences': 1}}  •  **embedding**: {'type': 'sentence_transformer', 'config': {'model_name': 'BAAI/bge-large-en-v1.5', 'model': None, 'dimension': 1024}}  •  **vector_store**: {'type': 'faiss', 'config': {'dimension': 1024}}  •  **retrieval**: {'search': {'searches': [{'type': 'dense', 'config': {'top_k': 20}}]}, 'query_transform': None, 'fusion': None, 'rerank': {'type': 'cross_encoder', 'config': {'model_name': 'BAAI/bge-reranker-v2-m3', 'top_k': 5}}}  •  **generation**: {'strategy': 'default', 'provider': 'groq', 'config': {'model': 'llama-3.1-8b-instant', 'max_tokens': 1024, 'temperature': 0.7, 'system_prompt': None}}  •  **evaluation**: {'type': 'trace', 'provider': 'groq', 'config': {'model': 'llama-3.1-8b-instant', 'max_tokens': 2000, 'temperature': 0.0}}  •  **cache**: {'enabled': True, 'cache_dir': './cache'}

### Per-query results

,query,retrieved_documents,answer,relevance_score__pred,relevance_score__gt,relevance_score__deviation,utilization_score__pred,utilization_score__gt,utilization_score__deviation,completeness_score__pred,completeness_score__gt,completeness_score__deviation,adherence_score__pred,adherence_score__gt,adherence_score__deviation
0,Which viruses may not cause prolonged inflammation due to strong induction o...,[{'text': 'successful treatment for HCV serves to circumvent the viral inhib...,"Based on the context provided, the viruses that may not cause prolonged infl...",0.1304,0.4118,-0.2814,0.1304,0.1765,-0.0461,1.0000,0.4286,0.5714,0.0,0.0,0.0
1,When was the first case of COVID-19 identified?,"[{'text': 'Abstract: In the WHO European Region, COVID-19 surveillance was i...","Unfortunately, this information isn't explicitly mentioned in the provided d...",0.0714,0.2692,-0.1978,0.0714,0.0769,-0.0055,1.0000,0.2857,0.7143,0.0,1.0,-1.0
2,How many antigens could be detected by Liew's multiplex ELISA test?,[{'text': 'Antibody arrays for simultaneous multiple antigen quantification ...,"According to Document 1 and Document 2, Liew validated one multiplex ELISA f...",0.6471,0.1250,0.5221,0.1176,0.0625,0.0551,0.1818,0.5000,-0.3182,1.0,1.0,0.0
3,What is the structure of Hantaan virus?,"[{'text': 'The diameter of hantavirus particles is 80-US210 nm, and the stru...",The structure of Hantaan virus is not explicitly mentioned in the provided c...,1.0000,0.3000,0.7000,0.1176,0.3000,-0.1824,0.1176,1.0000,-0.8824,0.0,1.0,-1.0
4,How many people did SARS-CoV infect?,[{'text': 'countries between 7 and 43 million individuals have been infected...,"According to Document 3, SARS-CoV caused 8096 reported cases in 27 countries.",0.1000,0.0667,0.0333,0.0500,0.0667,-0.0167,0.5000,1.0000,-0.5000,0.0,1.0,-1.0


### Aggregate TRACe scores (mean / ground truth / deviation)

,metric,mean_score,mean_ground_truth,std_score,std_ground_truth,mean_abs_error
0,relevance_score,0.3898,0.2345,0.3718,0.1242,0.3469
1,utilization_score,0.0974,0.1365,0.0311,0.0919,0.0612
2,completeness_score,0.5599,0.6429,0.3820,0.2997,0.5973
3,adherence_score,0.2000,0.8000,0.4000,0.4000,0.6000
